In [2]:
import pandas as pd

# Load the filtered expression matrix (matching clinical samples)
expr_path = r"C:\Users\Neg\RNA_seq_python_GC\expression_matched.csv"
expr_df = pd.read_csv(expr_path, index_col=0)

# Show shape and preview
print("Expression matrix shape:", expr_df.shape)
expr_df.iloc[:5, :5]  # show first 5 genes and 5 samples


Expression matrix shape: (56678, 448)


,TCGA-VQ-A8PP-01,TCGA-HU-A4H6-01,TCGA-MX-A5UG-01,TCGA-BR-8077-01,TCGA-CG-5720-01
Ensembl_ID,,,,,
ENSG00000000003.15,10.731319,11.339850,11.294046,13.200439,11.009129
ENSG00000000005.6,0.000000,0.000000,2.321928,0.000000,1.584963
ENSG00000000419.13,10.871135,12.141469,10.924813,12.038919,10.906139
ENSG00000000457.14,10.100662,10.306062,10.535275,9.552669,8.968667
ENSG00000000460.17,9.025140,9.388017,8.921841,8.758223,8.330917


In [3]:
# Extract sample types from TCGA barcodes
sample_types = expr_df.columns.str.slice(13, 15)

# Create a group label Series
sample_labels = sample_types.map({'01': 'Tumor', '11': 'Normal'})

# Show value counts
print("Sample group counts:")
print(sample_labels.value_counts())


Sample group counts:
Tumor     412
Normal     36
Name: count, dtype: int64


In [4]:
# Load clinical and expression files
clinical = pd.read_csv(r"C:\Users\Neg\RNA_seq_python_GC\clinical_cleaned.csv")
expr = pd.read_csv(r"C:\Users\Neg\RNA_seq_python_GC\filtered_counts.csv", index_col=0)

# Keep original column names for matching
common_ids = list(set(clinical['sample_id']) & set(expr.columns))

# Subset both datasets
expr_matched = expr[common_ids].copy()
clinical_matched = clinical[clinical['sample_id'].isin(common_ids)].copy()

# Now standardize sample barcodes in expr_matched
expr_matched.columns = expr_matched.columns.str.slice(0, 15)

# Save the matched files
expr_matched.to_csv(r"C:\Users\Neg\RNA_seq_python_GC\expression_matched.csv")
clinical_matched.to_csv(r"C:\Users\Neg\RNA_seq_python_GC\clinical_matched.csv", index=False)

print("✅ Fixed and saved matched expression and clinical files.")


✅ Fixed and saved matched expression and clinical files.


In [7]:
import pandas as pd

# Load the matched expression data
expr_df = pd.read_csv(r"C:\Users\Neg\RNA_seq_python_GC\expression_matched.csv", index_col=0)

# Preview shape and column names
print("Expression matrix shape:", expr_df.shape)
expr_df.iloc[:5, :5]  # first 5 genes x 5 samples


Expression matrix shape: (56678, 448)


,TCGA-BR-A4QM-01,TCGA-VQ-A8PP-01,TCGA-VQ-AA6I-01,TCGA-BR-A4PD-01,TCGA-VQ-A8PH-01
Ensembl_ID,,,,,
ENSG00000000003.15,10.700440,10.731319,10.427313,10.310613,12.659550
ENSG00000000005.6,0.000000,0.000000,2.321928,0.000000,1.000000
ENSG00000000419.13,10.663558,10.871135,11.401946,13.305634,12.694358
ENSG00000000457.14,8.459432,10.100662,10.766529,10.105909,9.287712
ENSG00000000460.17,6.584963,9.025140,10.283088,9.552669,9.467606


In [8]:
# Extract sample type from barcode (13–15 characters)
sample_types = expr_df.columns.str.slice(13, 15)

# Create tumor/normal group labels
sample_labels = sample_types.map({'01': 'Tumor', '11': 'Normal'})

# Check distribution
print("Sample group counts:")
print(sample_labels.value_counts())


Sample group counts:
Tumor     412
Normal     36
Name: count, dtype: int64


In [14]:
pip install rpy2

Note: you may need to restart the kernel to use updated packages.


In [11]:
%load_ext rpy2.ipython


C:\Users\Neg\anaconda3\Lib\site-packages\rpy2\robjects\packages.py:367: UserWarning: The symbol 'quartz' is not in this R namespace/package.
  warnings.warn(


In [12]:
%load_ext rpy2.ipython

%%R
R.version.string  # This will confirm your R setup


C:\Users\Neg\anaconda3\Lib\site-packages\rpy2\robjects\packages.py:367: UserWarning: The symbol 'quartz' is not in this R namespace/package.
  warnings.warn(
UsageError: Line magic function `%%R` not found.


In [13]:
%%R
R.version.string
S

[1] "R version 4.0.2 (2020-06-22)"


In [15]:
%%R
if (!requireNamespace("DESeq2", quietly = TRUE)) {
    install.packages("BiocManager")
    BiocManager::install("DESeq2")
}
library(DESeq2)


--- Please select a CRAN mirror for use in this session ---

  There is a binary version available but the source version is later:
             binary  source needs_compilation
BiocManager 1.30.16 1.30.25             FALSE

Update all/some/none? [a/s/n]: 

 n


Installing package into 'C:/Users/Neg/OneDrive/Documents/R/win-library/4.0'
(as 'lib' is unspecified)
installing the source package 'BiocManager'

trying URL 'https://cloud.r-project.org/src/contrib/BiocManager_1.30.25.tar.gz'
Content type 'application/x-gzip' length 593414 bytes (579 KB)
downloaded 579 KB


The downloaded source packages are in
	'C:\Users\Neg\AppData\Local\Temp\RtmpwxG1Gb\downloaded_packages'
'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org
Bioconductor version 3.12 (BiocManager 1.30.25), R 4.0.2 (2020-06-22)
Installation paths not writeable, unable to update packages
  path: C:/Program Files/R/R-4.0.2/library
  packages:
    boot, class, cluster, codetools, foreign, KernSmooth, lattice, mgcv, nlme,
    nnet, rpart, spatial, survival
Old packages: 'affy', 'affyio', 'annotate', 'AnnotationDbi', 'askpass',
  'backports', 'BH',

RInterpreterError: Failed to parse and evaluate line 'if (!requireNamespace("DESeq2", quietly = TRUE)) {\n    install.packages("BiocManager")\n    BiocManager::install("DESeq2")\n}\nlibrary(DESeq2)\n'.
R error message: "Error: package or namespace load failed for 'DESeq2' in loadNamespace(i, c(lib.loc, .libPaths()), versionCheck = vI[[i]]):\n object 'pkgInfo' not found"
R stdout:
Installing package into 'C:/Users/Neg/OneDrive/Documents/R/win-library/4.0'
(as 'lib' is unspecified)
installing the source package 'BiocManager'

trying URL 'https://cloud.r-project.org/src/contrib/BiocManager_1.30.25.tar.gz'
Content type 'application/x-gzip' length 593414 bytes (579 KB)
downloaded 579 KB


The downloaded source packages are in
	'C:\Users\Neg\AppData\Local\Temp\RtmpwxG1Gb\downloaded_packages'
'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org
Biocondu

In [17]:
%%R
install.packages("Rcpp", dependencies = TRUE, force = TRUE)



  There are binary versions available but the source versions are later:
           binary source needs_compilation
tinytest    1.3.1  1.4.1             FALSE
inline     0.3.19 0.3.21             FALSE
pkgKitten   0.2.2  0.2.4             FALSE
Rcpp      1.0.8.3 1.0.14              TRUE

  Binaries will be installed
package 'rbenchmark' successfully unpacked and MD5 sums checked
package 'Rcpp' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Neg\AppData\Local\Temp\RtmpwxG1Gb\downloaded_packages


Installing package into 'C:/Users/Neg/OneDrive/Documents/R/win-library/4.0'
(as 'lib' is unspecified)
also installing the dependencies 'tinytest', 'inline', 'rbenchmark', 'pkgKitten'

trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/rbenchmark_1.0.0.zip'
Content type 'application/zip' length 22292 bytes (21 KB)
downloaded 21 KB

trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/Rcpp_1.0.8.3.zip'
Content type 'application/zip' length 3354808 bytes (3.2 MB)
downloaded 3.2 MB

installing the source packages 'tinytest', 'inline', 'pkgKitten'

trying URL 'https://cloud.r-project.org/src/contrib/tinytest_1.4.1.tar.gz'
Content type 'application/x-gzip' length 587352 bytes (573 KB)
downloaded 573 KB

trying URL 'https://cloud.r-project.org/src/contrib/inline_0.3.21.tar.gz'
Content type 'application/x-gzip' length 25454 bytes (24 KB)
downloaded 24 KB

trying URL 'https://cloud.r-project.org/src/contrib/pkgKitten_0.2.4.tar.gz'
Content type 'application/x-gzip' length

In [19]:
%%R
BiocManager::install("DESeq2", force = TRUE)



  There are binary versions available but the source versions are later:
          binary source needs_compilation
lifecycle  1.0.1  1.0.4             FALSE
pillar     1.7.0 1.10.2             FALSE
cli        3.2.0  3.6.4              TRUE
glue       1.6.2  1.8.0              TRUE
gtable     0.3.0  0.3.6             FALSE
rlang      1.0.2  1.1.6              TRUE
scales     1.2.0  1.3.0              TRUE
tibble     3.1.6  3.2.1              TRUE
vctrs      0.4.1  0.6.5              TRUE
ggplot2    3.3.5  3.5.2             FALSE

  Binaries will be installed
package 'cli' successfully unpacked and MD5 sums checked
package 'glue' successfully unpacked and MD5 sums checked
package 'rlang' successfully unpacked and MD5 sums checked
package 'scales' successfully unpacked and MD5 sums checked
package 'tibble' successfully unpacked and MD5 sums checked
package 'vctrs' successfully unpacked and MD5 sums checked
package 'DESeq2' successfully unpacked and MD5 sums checked

The downloaded binar

 n


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org
Bioconductor version 3.12 (BiocManager 1.30.25), R 4.0.2 (2020-06-22)
Installing package(s) 'DESeq2'
also installing the dependencies 'lifecycle', 'pillar', 'cli', 'glue', 'gtable', 'rlang', 'scales', 'tibble', 'vctrs', 'ggplot2'

trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/cli_3.2.0.zip'
Content type 'application/zip' length 1255499 bytes (1.2 MB)
downloaded 1.2 MB

trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/glue_1.6.2.zip'
Content type 'application/zip' length 171950 bytes (167 KB)
downloaded 167 KB

trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/rlang_1.0.2.zip'
Content type 'application/zip' length 1718546 bytes (1.6 MB)
downloaded 1.6 MB

trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/scales_1.2.0.zip'
Content type 

In [21]:
%%R
library(DESeq2)


The package `ellipsis` (>= 0.3.2) is required as of rlang 1.0.0.
! Would you like to update it now?
  You will likely need to restart R if you update now.
i Not updating now is completely safe and will only cause import warnings.

1: Yes
2: No



Selection:  1


package 'ellipsis' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Neg\AppData\Local\Temp\RtmpwxG1Gb\downloaded_packages
Installing package into 'C:/Users/Neg/OneDrive/Documents/R/win-library/4.0'
(as 'lib' is unspecified)
trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/ellipsis_0.3.2.zip'
Content type 'application/zip' length 48927 bytes (47 KB)
downloaded 47 KB

Error: package or namespace load failed for 'DESeq2' in loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]):
 object 'pkgInfo' not found
In addition: Warning messages:
1: package 'DESeq2' was built under R version 4.0.3 
2: In loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]) :
  package 'memoise' has no 'package.rds' in Meta/
Error: package or namespace load failed for 'DESeq2' in loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]):
 object 'pkgInfo' not found


RInterpreterError: Failed to parse and evaluate line 'library(DESeq2)\n'.
R error message: "Error: package or namespace load failed for 'DESeq2' in loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]):\n object 'pkgInfo' not found"
R stdout:
Installing package into 'C:/Users/Neg/OneDrive/Documents/R/win-library/4.0'
(as 'lib' is unspecified)
trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/ellipsis_0.3.2.zip'
Content type 'application/zip' length 48927 bytes (47 KB)
downloaded 47 KB

Error: package or namespace load failed for 'DESeq2' in loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]):
 object 'pkgInfo' not found
In addition: Warning messages:
1: package 'DESeq2' was built under R version 4.0.3 
2: In loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]) :
  package 'memoise' has no 'package.rds' in Meta/

In [23]:
%%R
install.packages("BiocManager", repos = "https://cloud.r-project.org")
BiocManager::install("DESeq2", force = TRUE)



  There is a binary version available but the source version is later:
             binary  source needs_compilation
BiocManager 1.30.16 1.30.25             FALSE


  There are binary versions available but the source versions are later:
          binary source needs_compilation
cli        3.2.0  3.6.4              TRUE
lifecycle  1.0.1  1.0.4             FALSE
gtable     0.3.0  0.3.6             FALSE
rlang      1.0.2  1.1.6              TRUE
scales     1.2.0  1.3.0              TRUE
vctrs      0.4.1  0.6.5              TRUE
ggplot2    3.3.5  3.5.2             FALSE

  Binaries will be installed
package 'cli' successfully unpacked and MD5 sums checked
package 'rlang' successfully unpacked and MD5 sums checked
package 'scales' successfully unpacked and MD5 sums checked
package 'vctrs' successfully unpacked and MD5 sums checked
package 'DESeq2' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Neg\AppData\Local\Temp\RtmpwxG1Gb\downloaded_packag

 a



  There are binary versions available but the source versions are later:
                  binary    source needs_compilation
pkgbuild           1.3.1     1.4.7             FALSE
dplyr              1.0.8     1.1.4              TRUE
curl               4.3.2     6.2.2              TRUE
stringi            1.7.6     1.8.7              TRUE
callr              3.7.0     3.7.6             FALSE
pkgload            1.2.4     1.4.0             FALSE
askpass              1.1     1.2.1              TRUE
backports          1.4.1     1.5.0              TRUE
BH              1.78.0-0  1.87.0-1             FALSE
bit                4.0.4     4.6.0              TRUE
bit64              4.0.5   4.6.0-1              TRUE
bitops             1.0-7     1.0-9              TRUE
blob               1.2.3     1.2.4             FALSE
brio               1.1.3     1.1.5              TRUE
cachem             1.0.6     1.1.0              TRUE
cli                3.2.0     3.6.4              TRUE
colorspace         2.0-3 

RInterpreterError: Failed to parse and evaluate line 'install.packages("BiocManager", repos = "https://cloud.r-project.org")\nBiocManager::install("DESeq2", force = TRUE)\n'.
R error message: "Error in unpackPkgZip(foundpkgs[okp, 2L], foundpkgs[okp, 1L], lib, libs_only,  : \n  ERROR: failed to lock directory 'C:\\Users\\Neg\\OneDrive\\Documents\\R\\win-library\\4.0' for modifying\nTry removing 'C:\\Users\\Neg\\OneDrive\\Documents\\R\\win-library\\4.0/00LOCK'"
R stdout:
Installing package into 'C:/Users/Neg/OneDrive/Documents/R/win-library/4.0'
(as 'lib' is unspecified)
installing the source package 'BiocManager'

trying URL 'https://cloud.r-project.org/src/contrib/BiocManager_1.30.25.tar.gz'
Content type 'application/x-gzip' length 593414 bytes (579 KB)
downloaded 579 KB


The downloaded source packages are in
	'C:\Users\Neg\AppData\Local\Temp\RtmpwxG1Gb\downloaded_packages'
'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManag

In [14]:
import rpy2
print(rpy2.__version__)


3.5.17


In [16]:
import rpy2.robjects as robjects
print(robjects.r('R.home()'))


R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library 'C:/Users/Neg/R_libs' contains no packages



[1] "C:/Program Files/R/R-4.0.2"



In [18]:
import rpy2.ipython


In [20]:
import sys
print(sys.executable)


C:\Users\Neg\anaconda3\python.exe


In [12]:
import rpy2.robjects.packages as rpackages
from rpy2.robjects.vectors import StrVector

# Ensure BiocManager is installed
utils = rpackages.importr('utils')
utils.chooseCRANmirror(ind=1)  # select the first CRAN mirror

if not rpackages.isinstalled("BiocManager"):
    utils.install_packages("BiocManager")

# Use BiocManager to install DESeq2
biocmanager = rpackages.importr("BiocManager")
biocmanager.install("DESeq2")


R[write to console]: Installing package into 'C:/Users/Neg/R_libs'
(as 'lib' is unspecified)




  There is a binary version available but the source version is later:
             binary  source needs_compilation
BiocManager 1.30.16 1.30.25             FALSE



R[write to console]: installing the source package 'BiocManager'


R[write to console]: trying URL 'https://cloud.r-project.org/src/contrib/BiocManager_1.30.25.tar.gz'

R[write to console]: Content type 'application/x-gzip'
R[write to console]:  length 593414 bytes (579 KB)

R[write to console]: downloaded 579 KB


R[write to console]: 

R[write to console]: 
R[write to console]: The downloaded source packages are in
	'C:\Users\Neg\AppData\Local\Temp\RtmpwVZ5RG\downloaded_packages'
R[write to console]: 
R[write to console]: 

R[write to console]: 'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

R[write to console]: Bioconductor version 3.12 (BiocManager 1.30.25), R 4.0.2 (2020-06-22)

R[write to console]: Installing package(s) 'BiocVersion', 'DESeq2'

R[write to console]: Warning:
R[write to console]:  dependency 'locfit' is not available

R


  There are binary versions available but the source versions are later:
                  binary    source needs_compilation
fastmap            1.1.0     1.2.0              TRUE
sys                  3.4     3.4.3              TRUE
bit                4.0.4     4.6.0              TRUE
cachem             1.0.6     1.1.0              TRUE
askpass              1.1     1.2.1              TRUE
bitops             1.0-7     1.0-9              TRUE
formatR             1.12      1.14             FALSE
bit64              4.0.5   4.6.0-1              TRUE
blob               1.2.3     1.2.4             FALSE
cpp11              0.4.2     0.5.2             FALSE
curl               4.3.2     6.2.2              TRUE
jsonlite           1.8.0     2.0.0              TRUE
mime                0.12      0.13              TRUE
openssl            2.0.0     2.3.2              TRUE
colorspace         2.0-3     2.1-1              TRUE
utf8               1.2.2     1.2.4              TRUE
RCurl           1.98-1.6 

R[write to console]: trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/fastmap_1.1.0.zip'

R[write to console]: Content type 'application/zip'
R[write to console]:  length 215436 bytes (210 KB)

R[write to console]: downloaded 210 KB


R[write to console]: trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/sys_3.4.zip'

R[write to console]: Content type 'application/zip'
R[write to console]:  length 59823 bytes (58 KB)

R[write to console]: downloaded 58 KB


R[write to console]: trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/bit_4.0.4.zip'

R[write to console]: Content type 'application/zip'
R[write to console]:  length 633925 bytes (619 KB)

R[write to console]: downloaded 619 KB


R[write to console]: trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.0/cachem_1.0.6.zip'

R[write to console]: Content type 'application/zip'
R[write to console]:  length 78814 bytes (76 KB)

R[write to console]: downloaded 76 KB


R[write to consol

package 'fastmap' successfully unpacked and MD5 sums checked
package 'sys' successfully unpacked and MD5 sums checked
package 'bit' successfully unpacked and MD5 sums checked
package 'cachem' successfully unpacked and MD5 sums checked
package 'askpass' successfully unpacked and MD5 sums checked
package 'bitops' successfully unpacked and MD5 sums checked
package 'bit64' successfully unpacked and MD5 sums checked
package 'memoise' successfully unpacked and MD5 sums checked
package 'plogr' successfully unpacked and MD5 sums checked
package 'curl' successfully unpacked and MD5 sums checked
package 'jsonlite' successfully unpacked and MD5 sums checked
package 'mime' successfully unpacked and MD5 sums checked
package 'openssl' successfully unpacked and MD5 sums checked
package 'colorspace' successfully unpacked and MD5 sums checked
package 'utf8' successfully unpacked and MD5 sums checked
package 'RCurl' successfully unpacked and MD5 sums checked
package 'zlibbioc' successfully unpacked and 

R[write to console]: installing the source packages 'formatR', 'blob', 'cpp11', 'GenomeInfoDbData', 'DBI', 'httr', 'labeling', 'munsell', 'R6', 'viridisLite', 'pillar', 'BH', 'gtable', 'lifecycle', 'withr', 'ggplot2'


R[write to console]: trying URL 'https://cloud.r-project.org/src/contrib/formatR_1.14.tar.gz'

R[write to console]: Content type 'application/x-gzip'
R[write to console]:  length 96077 bytes (93 KB)

R[write to console]: downloaded 93 KB


R[write to console]: trying URL 'https://cloud.r-project.org/src/contrib/blob_1.2.4.tar.gz'

R[write to console]: Content type 'application/x-gzip'
R[write to console]:  length 10620 bytes (10 KB)

R[write to console]: downloaded 10 KB


R[write to console]: trying URL 'https://cloud.r-project.org/src/contrib/cpp11_0.5.2.tar.gz'

R[write to console]: Content type 'application/x-gzip'
R[write to console]:  length 291993 bytes (285 KB)

R[write to console]: downloaded 285 KB


R[write to console]: trying URL 'https://bioconductor.org/pac

Update all/some/none? [a/s/n]: 

 n


'BiocVersion','DESeq2'


In [14]:
import rpy2.robjects as robjects

# Set the user-writable library path
robjects.r('.libPaths("C:/Users/Neg/R_libs")')

# Confirm the current library paths
print(robjects.r('.libPaths()'))


R[write to console]: In addition: 
R[write to console]: Warning messages:

R[write to console]: 1: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library 'C:/Users/Neg/R_libs' contains no packages

R[write to console]: 2: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library 'C:/Users/Neg/R_libs' contains no packages

R[write to console]: 3: 
R[write to console]: In install.packages(...) :
R[write to console]: 
 
R[write to console]:  installation of package 'lifecycle' had non-zero exit status

R[write to console]: 4: 
R[write to console]: In install.packages(...) :
R[write to console]: 
 
R[write to console]:  installation of package 'pillar' had non-zero exit status

R[write to console]: 5: 
R[write to console]: In install.packages(...) :
R[write to console]: 
 
R[write to console]

[1] "C:/Users/Neg/R_libs"                "C:/Program Files/R/R-4.0.2/library"

